In [ ]:
!python --version

Python 3.12.13


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")

In [ ]:
df.head(5)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [ ]:
df.sample(10)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
57124,11399,TomClancysRainbowSix,Neutral,ALRIGHT! SO great stream battle! Nothing to ex...
17484,9799,PlayStation5(PS5),Negative,"Oops, caught making people think it's a comple..."
47200,5698,HomeDepot,Positive,or
51867,10511,RedDeadRedemption(RDR),Positive,I enjoyed watching up this nice number one Rog...
35649,8121,Microsoft,Neutral,“ the Free Software Day movement is long dead....
44836,11699,Verizon,Positive,2.0 @CharityMiles for Windows. Thanks to insta...
47380,5728,HomeDepot,Positive,Today went so cool. I decided the give outdoor...
48502,5926,HomeDepot,Neutral,STOP HERE STEAL MY<unk> IN TO HOME O BYEEHEBSH...
21917,4151,CS-GO,Positive,Nice afternoon CS:GO hit thanks to the awesome...
34681,6757,Fortnite,Negative,My brother was scared.


In [ ]:
df.shape

(74681, 4)

In [ ]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [ ]:
df.describe()

,2401
count,74681.000000
mean,6432.640149
std,3740.423819
min,1.000000
25%,3195.000000
50%,6422.000000
75%,9601.000000
max,13200.000000


In [ ]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)

In [ ]:
df

,2401,Borderlands,sentiment,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [ ]:
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [ ]:
df.sample(5)

,2401,Borderlands,sentiment,text
57573,11474,TomClancysRainbowSix,Neutral,When released you have a certain 5stack that e...
38532,5409,Hearthstone,Irrelevant,I think @ Monsanto _ HS is good for Hearthston...
53451,10784,RedDeadRedemption(RDR),Irrelevant,Cant see clearly why you would willingly get i...
35998,8179,Microsoft,Positive,Trust<unk> Technology is Important!. @satyanad...
64717,7886,MaddenNFL,Positive,You're the devil so it doesn't matter


In [ ]:
df = df[['text', 'sentiment']]

In [ ]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [ ]:
df.shape

(74681, 2)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       73995 non-null  object
 1   sentiment  74681 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


In [ ]:
df.describe()

,text,sentiment
count,73995,74681
unique,69490,4
top,,Negative
freq,172,22542


In [ ]:
df.columns

Index(['text', 'sentiment'], dtype='object')

In [ ]:
df.isnull().sum()

,0
text,686
sentiment,0


In [ ]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [ ]:
df.duplicated().sum()

np.int64(4909)

In [ ]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_2837/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.dropna(inplace=True)

/tmp/ipykernel_2837/1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [ ]:
df.isnull().sum()

,0
text,0
sentiment,0


In [ ]:
df.shape

(69768, 2)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69768 entries, 0 to 74680
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       69768 non-null  object
 1   sentiment  69768 non-null  object
dtypes: object(2)
memory usage: 1.6+ MB


In [ ]:
df.describe()

,text,sentiment
count,69768,69768
unique,69490,4
top,by,Negative
freq,4,21237


In [ ]:
df.sample(10)

,text,sentiment
48980,He forgot to do his leg checking before the ra...,Negative
27607,"[PS4] ""Assassins Creed Syndic"" is the first fe...",Positive
26489,which assassins creed game was it where ubisof...,Negative
6564,"Warm, vulnerable and insightful! Grab a copy o...",Neutral
67681,Love the,Neutral
65143,It's amazing that they show his personality an...,Positive
19436,I just earned the Golden Vision of Orgrimmar] ...,Neutral
23283,Nuttiest black thing indeed I have ever ever s...,Positive
23876,I’ve gotten to feel really jumpy ever since Tw...,Negative
55301,Last night I had 3 viewers! I was excited.,Irrelevant


In [ ]:
df['text'] = df['text'].astype(str)

/tmp/ipykernel_2837/1180961456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].astype(str)


In [ ]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

/tmp/ipykernel_2837/1069262265.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(clean_text)


In [ ]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


In [43]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


/tmp/ipykernel_2837/3100372417.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])


In [44]:
encoder

LabelEncoder()

In [45]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [46]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [47]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="")
tokenizer.fit_on_texts(df['text'])

X = pad_sequences(
    tokenizer.texts_to_sequences(df['text']),
    maxlen=max_length,
    padding='post'
)

y = df['sentiment_encoded'].values

In [50]:
tokenizer

In [48]:
X

array([[   3,   98,  370, ...,    0,    0,    0],
       [  31,  155,   14, ...,    0,    0,    0],
       [  31,  370,   14, ...,    0,    0,    0],
       ...,
       [  21, 1830,    2, ...,    0,    0,    0],
       [  21, 1830,  683, ...,    0,    0,    0],
       [  21,   32,    2, ...,    0,    0,    0]], dtype=int32)

In [49]:
X.shape

(69768, 100)

In [51]:
y

array([3, 3, 3, ..., 3, 3, 3])

In [52]:
y.shape

(69768,)

In [53]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [54]:
X.dtype

dtype('int32')

In [55]:
y.dtype

dtype('int64')

In [56]:
X.ndim

2

In [57]:
y.ndim

1

In [58]:
X.nbytes

27907200

In [59]:
y.nbytes

558144